In [ ]:
import pandas as pd
import numpy as np
import folium
import joblib
from sklearn.cluster import HDBSCAN
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# chargement des données
data_file = "C:/Users/melis/Downloads/export_IA.csv"  
data = pd.read_csv(data_file, low_memory=False)

# tri des données "non-valides"
donnees_valides = data[data['consolidated_is_lon_lat_correct'] == True].copy()
X = np.array(list(zip(donnees_valides['consolidated_latitude'], donnees_valides['consolidated_longitude'])))

# lancement du modèle avec HDBSCAN
hdb_final = HDBSCAN(
    min_cluster_size=250, 
    min_samples=5, 
    cluster_selection_epsilon=0.12
)
donnees_valides["cluster_hdbscan"] = hdb_final.fit_predict(X)

# génération de la carte
centre_france = [donnees_valides["consolidated_latitude"].mean(), donnees_valides["consolidated_longitude"].mean()]
map_finale = folium.Map(location=centre_france, zoom_start=6)

# on affiche que 10 000 points pour la fluidité d'affichage
echantillon = donnees_valides.sample(min(10000, len(donnees_valides)), random_state=42)

# mise en place des couleurs
couleurs = {-1: 'black'} 
cmap = plt.get_cmap('tab20')

clusters_uniques = set(donnees_valides["cluster_hdbscan"]) 
index_couleur = 0

for cluster in clusters_uniques:
    if cluster != -1: 
        couleurs[cluster] = mcolors.to_hex(cmap(index_couleur % 20)) # Le % 20 évite les bugs si le nombre de clusters dépasse les 20 couleurs de la palette
        index_couleur += 1

# ajout des points sur la carte
for _, row in echantillon.iterrows():
    cluster_actuel = row["cluster_hdbscan"]
    couleur = couleurs.get(cluster_actuel, "gray") 
    
    folium.CircleMarker(
        location=[row["consolidated_latitude"], row["consolidated_longitude"]],
        radius=1.5 if cluster_actuel == -1 else 3,
        color=couleur,        
        fill=True,
        fill_color=couleur,   
        fill_opacity=0.4 if cluster_actuel == -1 else 0.7
    ).add_to(map_finale) 

map_finale.save("carte_hdbsan.html")

# enregistrement du modèle
joblib.dump(hdb_final, "modele_hdbscan_optimal.pkl")
print("Fichiers cartographiques et modèle sauvegardés")

Exécution terminée. Fichiers cartographiques et modèle sauvegardés avec succès !
